In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
raw_data = project_root / "data" / "raw" / "customer_support_tickets.csv"
db = project_root / "data" / "processed" / "customer_support.db"

### Запрос 1. Тикеты по каналу Email, отсортированные по времени первого ответа
Выводит `ticket_id`, `ticket_channel` и `first_response_time` для тикетов с каналом `Email`, 
отсортированные по времени первого ответа от самого раннего к самому позднему.

**Конструкции:** SELECT, WHERE, ORDER BY

In [5]:
q1 = '''
SELECT ticket_id, ticket_channel, first_response_time
FROM tickets
WHERE ticket_channel = "Email"
ORDER BY first_response_time;
'''
with sqlite3.connect(db) as conn:
    result1 = pd.read_sql(q1, conn)

result1

,ticket_id,ticket_channel,first_response_time
0,25,Email,NaN
1,31,Email,NaN
2,44,Email,NaN
3,58,Email,NaN
4,75,Email,NaN
...,...,...,...
2138,7757,Email,2023-06-02 00:12:48
2139,7311,Email,2023-06-02 00:14:21
2140,7013,Email,2023-06-02 00:15:37
2141,8076,Email,2023-06-02 00:28:24


### Запрос 2. Количество тикетов и средний рейтинг по приоритету
Для каждого `ticket_priority` считает количество тикетов и средний `customer_satisfaction_rating`. 
Оставляет только приоритеты, где тикетов больше 500.

**Конструкции:** SELECT, GROUP BY, HAVING, агрегатные функции (COUNT, AVG)

In [ ]:
q2 = '''
SELECT
    ticket_priority, COUNT(*) as count_for_priority, AVG(customer_satisfaction_rating) as avg_rating
FROM tickets 
GROUP BY ticket_priority
HAVING COUNT(*) > 500
'''

with sqlite3.connect(db) as conn:
    result2 = pd.read_sql(q2, conn)

result2

,ticket_priority,count_for_priority,acg_rating
0,Critical,2129,2.958678
1,High,2085,2.982979
2,Low,2063,3.052795
3,Medium,2192,2.976945


### Запрос 3. Категоризация тикетов по уровню удовлетворенности
Присваивает каждому тикету категорию (`High` / `Medium` / `Low` / `No rating`) 
на основе значения `customer_satisfaction_rating`.

**Конструкции:** SELECT, CASE

In [8]:
q3 = '''
SELECT
    ticket_id, customer_satisfaction_rating,
    CASE
        WHEN customer_satisfaction_rating BETWEEN 4 AND 5 THEN "High"
        WHEN customer_satisfaction_rating BETWEEN 2 AND 3 THEN "Medium"
        WHEN customer_satisfaction_rating = 1 THEN "Bad"
        ELSE "No rating"
    END as satisfaction_level
FROM tickets;

'''

with sqlite3.connect(db) as conn:
    result3 = pd.read_sql(q3, conn)

result3

,ticket_id,customer_satisfaction_rating,satisfaction_level
0,1,NaN,No rating
1,2,NaN,No rating
2,3,3.0,Medium
3,4,3.0,Medium
4,5,1.0,Bad
...,...,...,...
8464,8465,NaN,No rating
8465,8466,NaN,No rating
8466,8467,3.0,Medium
8467,8468,3.0,Medium


### Запрос 4. Продукт и тип обращения по каждому тикету
Соединяет `tickets` с `products` и `ticket_types`, чтобы вывести название продукта 
и тип обращения вместо их числовых ID. Первые 10 строк.

**Конструкции:** SELECT, JOIN (несколько таблиц), LIMIT

In [9]:
q4 = '''
SELECT 
    t.ticket_id, p.product_name, tt.ticket_type_name
FROM tickets t
JOIN products p ON p.product_id = t.product_id
JOIN ticket_types tt ON tt.ticket_type_id = t.ticket_type_id
LIMIT 10;
'''

with sqlite3.connect(db) as conn:
    result4 = pd.read_sql(q4, conn)

result4

,ticket_id,product_name,ticket_type_name
0,1,GoPro Hero,Technical issue
1,2,LG Smart TV,Technical issue
2,3,Dell XPS,Technical issue
3,4,Microsoft Office,Billing inquiry
4,5,Autodesk AutoCAD,Billing inquiry
5,6,Microsoft Office,Cancellation request
6,7,Microsoft Surface,Product inquiry
7,8,Philips Hue Lights,Refund request
8,9,Fitbit Versa Smartwatch,Technical issue
9,10,Dyson Vacuum Cleaner,Refund request


### Запрос 5. Топ 10 продуктов по количеству обращений
Считает количество тикетов по каждому продукту, сортирует по убыванию 
и показывает 10 продуктов с наибольшим числом обращений.

**Конструкции:** SELECT, JOIN, GROUP BY, ORDER BY, LIMIT

In [10]:
q5 = '''
    SELECT p.product_name, COUNT(*) as product_count
    FROM tickets t
    JOIN products p ON p.product_id = t.product_id
    GROUP BY product_name
    ORDER BY COUNT(*) DESC
    LIMIT 10;
'''

with sqlite3.connect(db) as conn:
    result5 = pd.read_sql(q5, conn)

result5

,product_name,product_count
0,Canon EOS,240
1,GoPro Hero,228
2,Nest Thermostat,225
3,Philips Hue Lights,221
4,Amazon Echo,221
5,LG Smart TV,219
6,Sony Xperia,217
7,Roomba Robot Vacuum,216
8,LG OLED,213
9,Apple AirPods,213


### Запрос 6. Критичные тикеты по телефону
Выводит тикеты с приоритетом `Critical`, поступившие по каналу `Phone`.

**Конструкции:** SELECT, WHERE (несколько условий через AND)

In [11]:
q6 = '''
    SELECT 
        ticket_id, ticket_priority, ticket_channel
    FROM tickets
    WHERE ticket_priority = 'Critical' AND ticket_channel = 'Phone';
'''

with sqlite3.connect(db) as conn:
    result6 = pd.read_sql(q6, conn)

result6

,ticket_id,ticket_priority,ticket_channel
0,10,Critical,Phone
1,29,Critical,Phone
2,39,Critical,Phone
3,77,Critical,Phone
4,81,Critical,Phone
...,...,...,...
521,8372,Critical,Phone
522,8414,Critical,Phone
523,8427,Critical,Phone
524,8439,Critical,Phone


### Запрос 7. Тикеты в статусе Closed или Pending Customer Response
Выводит `ticket_id` и `ticket_status` для тикетов, находящихся в одном из двух статусов.

**Конструкции:** SELECT, WHERE, IN

In [12]:
q7 = '''
    SELECT t.ticket_id, t.ticket_status
    FROM tickets t
    WHERE ticket_status IN ("Closed", "Pending Customer Response")
'''

with sqlite3.connect(db) as conn:
    result7 = pd.read_sql(q7, conn)

result7

,ticket_id,ticket_status
0,1,Pending Customer Response
1,2,Pending Customer Response
2,3,Closed
3,4,Closed
4,5,Closed
...,...,...
5645,8456,Closed
5646,8459,Pending Customer Response
5647,8463,Pending Customer Response
5648,8467,Closed


### Запрос 8. Количество тикетов по группе каналов (онлайн/оффлайн)
Группирует каналы обращения на "Онлайн" (Email, Chat, Social media) и "Оффлайн" (Phone), 
считает количество тикетов в каждой группе.

**Конструкции:** SELECT, CASE, GROUP BY

In [ ]:
q8 = '''
    SELECT
        CASE
            WHEN ticket_channel IN ('Email', 'Chat', 'Social media') THEN 'Онлайн'
            WHEN ticket_channel = 'Phone' THEN 'Оффлайн'
        END as channel_group,
        COUNT(*) as tickets_count
    FROM tickets
    GROUP BY channel_group;
'''

with sqlite3.connect(db) as conn:
    result8 = pd.read_sql(q8, conn)

result8

### Запрос 9. Возрастной диапазон клиентов с критичными тикетами
Находит минимальный и максимальный возраст среди клиентов, 
у которых был хотя бы один тикет с приоритетом Critical.

**Конструкции:** SELECT, JOIN, WHERE, агрегатные функции без GROUP BY (MIN, MAX)

In [ ]:
q9 = '''
    SELECT
        MIN(c.customer_age) as min_age,
        MAX(c.customer_age) as max_age
    FROM tickets t
    JOIN customers c ON t.customer_id = c.customer_id
    WHERE t.ticket_priority = 'Critical';
'''
with sqlite3.connect(db) as conn:
    result9 = pd.read_sql(q9, conn)

result9

### Запрос 10. Каналы с низким средним рейтингом удовлетворенности
Для каждого канала обращения считает количество тикетов и средний рейтинг удовлетворенности, 
оставляет только каналы со средним рейтингом ниже 3.

**Конструкции:** SELECT, GROUP BY, HAVING, агрегатные функции (COUNT, AVG)

In [ ]:
q10 = '''
    SELECT
        ticket_channel,
        COUNT(*) as tickets_count,
        AVG(customer_satisfaction_rating) as avg_rating
    FROM tickets
    GROUP BY ticket_channel
    HAVING AVG(customer_satisfaction_rating) < 3;
'''
with sqlite3.connect(db) as conn:
    result10 = pd.read_sql(q10, conn)

result10